# Project 3: Data Cleaning - Tidy up messy Datasets (Movies Dataset)

# Project Brief for Self-Coders

Here you´ll have the opportunity to code major parts of Project 3 on your own. If you need any help or inspiration, have a look at the Videos or the Jupyter Notebook with the full code. <br> <br>
Keep in mind that it´s all about __getting the right results/conclusions__. It´s not about finding the identical code. Things can be coded in many different ways. Even if you come to the same conclusions, it´s very unlikely that we have the very same code. 

## First Steps 

1. __Load__ and __inspect__ the messy dataset __movies_metadata.csv__. Identify columns with nested / stringified json data.

In [167]:
import pandas as pd
import numpy as np
import ast

In [168]:
movies = pd.read_csv('movies_metadata.csv')
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

C:\Users\ahmed\AppData\Local\Temp\ipykernel_16804\3865073641.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv('movies_metadata.csv')


In [169]:
movies.describe()

,revenue,runtime,vote_average,vote_count
count,4.546000e+04,45203.000000,45460.000000,45460.000000
mean,1.120935e+07,94.128199,5.618207,109.897338
std,6.433225e+07,38.407810,1.924216,491.310374
min,0.000000e+00,0.000000,0.000000,0.000000
25%,0.000000e+00,85.000000,5.000000,3.000000
50%,0.000000e+00,95.000000,6.000000,10.000000
75%,0.000000e+00,107.000000,6.800000,34.000000
max,2.787965e+09,1256.000000,10.000000,14075.000000


In [170]:
movies.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

In [171]:
movies[["title",'video','homepage','original_title','status']].head()

,title,video,homepage,original_title,status
0,Toy Story,False,http://toystory.disney.com/toy-story,Toy Story,Released
1,Jumanji,False,NaN,Jumanji,Released
2,Grumpier Old Men,False,NaN,Grumpier Old Men,Released
3,Waiting to Exhale,False,NaN,Waiting to Exhale,Released
4,Father of the Bride Part II,False,NaN,Father of the Bride Part II,Released


## Dropping irrelevant Columns

2. __Drop__ the irrelevant columns 'adult', 'imdb_id', 'original_title', 'video' and 'homepage'.

In [172]:
columns_to_be_dropped = ['adult','imdb_id','original_title',"video",'homepage']
movies = movies.drop(columns=columns_to_be_dropped)


## How to handle stringified JSON columns

3. __Evaluate__ Python Expressions in the stringified columns ["belongs_to_collection", "genres", "production_countries", "production_companies", "spoken_languages"] and __remove quotes__ ("") where possible.

In [173]:
print(movies[["belongs_to_collection", "genres", "production_countries", "production_companies", "spoken_languages"]].head())

                               belongs_to_collection  \
0  {'id': 10194, 'name': 'Toy Story Collection', ...   
1                                                NaN   
2  {'id': 119050, 'name': 'Grumpy Old Men Collect...   
3                                                NaN   
4  {'id': 96871, 'name': 'Father of the Bride Col...   

                                              genres  \
0  [{'id': 16, 'name': 'Animation'}, {'id': 35, '...   
1  [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...   
2  [{'id': 10749, 'name': 'Romance'}, {'id': 35, ...   
3  [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...   
4                     [{'id': 35, 'name': 'Comedy'}]   

                                production_countries  \
0  [{'iso_3166_1': 'US', 'name': 'United States o...   
1  [{'iso_3166_1': 'US', 'name': 'United States o...   
2  [{'iso_3166_1': 'US', 'name': 'United States o...   
3  [{'iso_3166_1': 'US', 'name': 'United States o...   
4  [{'iso_3166_1': 'US', 'name': 'United State

In [174]:
movies.genres[0]

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [175]:
movies['genres'] = movies['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Extract genre names and join them with '|'
movies['genres'] = movies['genres'].apply(
    lambda x: '|'.join([d['name'] for d in x]) if isinstance(x, list) else ''
)

In [176]:
movies['genres'].head()

0     Animation|Comedy|Family
1    Adventure|Fantasy|Family
2              Romance|Comedy
3        Comedy|Drama|Romance
4                      Comedy
Name: genres, dtype: object

In [177]:
movies['belongs_to_collection'][1]

nan

In [178]:
movies['belongs_to_collection'] = movies['belongs_to_collection'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
movies['belongs_to_collection'] = movies['belongs_to_collection'].apply(lambda x: x['name'] if isinstance(x, dict) and pd.notna(x.get('name')) else 'Standalone')
movies['belongs_to_collection'].head()

0              Toy Story Collection
1                        Standalone
2         Grumpy Old Men Collection
3                        Standalone
4    Father of the Bride Collection
Name: belongs_to_collection, dtype: object

In [179]:
movies['production_companies'] = movies['production_companies'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
movies['production_companies'] = movies['production_companies'].apply(lambda x: '|'.join([d['name'] for d in x]) if isinstance(x, list) else '')


In [180]:
movies['production_countries'] = movies['production_countries'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

movies['production_countries'] = movies['production_countries'].apply(lambda x: '|'.join([d['name'] for d in x]) if isinstance(x, list) else '')
movies['production_countries'][0]


'United States of America'

In [181]:
movies['spoken_languages'] = movies['spoken_languages'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
movies['spoken_languages'] = movies['spoken_languages'].apply(lambda x: '|'.join([d['name'] for d in x]) if isinstance(x, list) else '')
movies['spoken_languages'].head()

0             English
1    English|Français
2             English
3             English
4             English
Name: spoken_languages, dtype: object

## How to flatten nested Columns

4. __Extract__ only the __collection name__ from the column "belongs_to_collection" and __overwrite__ "belongs_to_collection". <br> For example: The value in the first row (Toy Story) should be 'Toy Story Collection'.

5. __Extract__ all __genre names__ from the column "genres" and __overwrite__ "genres". If a movie has more than one genre, __seperate genres by a pipe__ "|".<br>
For example: The value in the first row (Toy Story) should be 'Animation|Comedy|Family'.

6. __Extract__ all __spoken language names__ from the column "spoken_languages" and __overwrite__ "spoken_languages". If a movie has more than one spoken language, __seperate spoken languages by a pipe__ "|".<br>
For example: The value in the first row (Toy Story) should be 'English'.

7. __Extract__ all __production countries names__ from the column "production_countries" and __overwrite__ "production_countries". If a movie has more than one production country, __seperate production countries by a pipe__ "|".<br>
For example: The value in the first row (Toy Story) should be 'United States of America'.

8. __Extract__ all __production companies names__ from the column "production_companies" and __overwrite__ "production_companies". If a movie has more than one production company, __seperate production companies by a pipe__ "|".<br>
For example: The value in the first row (Toy Story) should be 'Pixar Animation Studios'

9. __Inspect__ all columns above with value_counts(). Do you see anything strange? __Take reasonable measures__!

In [182]:
movies[["belongs_to_collection", "genres", "production_countries", "production_companies", "spoken_languages"]].value_counts()

belongs_to_collection  genres                production_countries                                    production_companies                                                                                     spoken_languages
Standalone                                                                                                                                                                                                                        1052
                       Documentary                                                                                                                                                                                                 456
                                                                                                                                                                                                              English              338
                                             United States of America               

## Cleaning Numerical Columns

10. __Convert__ the datatype in the columns __"budget"__, __"id"__ and __"popularity"__ __to numeric__. Set invalid values as NaN.

11. __Analyze__ the columns __"budget"__ and __"revenue"__ and __"runtime"__. Analyze movies with a budget/revenue/runtime of 0. Do you think the value 0 is the most appropriate value? __Take reasonable measures__! 

12. The columns "budget" and "revenue" shall show values in Million USD. __Convert and Overwrite__!

13. __Analyze__ movies with a __vote_count of 0__. What´s the __vote_average__ for those movies? Do you think this value is the most appropriate value? __Take reasonable measures__!

In [183]:
col = ['budget','id','popularity']
movies[col] = movies[col].apply(pd.to_numeric,errors ='coerce') 


In [186]:
cols = ['budget','revenue','runtime']
movies[cols] = movies[cols].replace(0,np.nan)
movies[cols] = movies[cols].apply(lambda col : col.fillna(col.median()))

In [188]:
cols = ['budget','revenue']
movies[cols] = movies[cols].div(1000000)


In [197]:
movies = movies.rename(columns={'budget':'budget_musd','revenue':'revenue_musd'})

In [198]:
movies.columns

Index(['belongs_to_collection', 'budget_musd', 'genres', 'id',
       'original_language', 'overview', 'popularity', 'poster_path',
       'production_companies', 'production_countries', 'release_date',
       'revenue_musd', 'runtime', 'spoken_languages', 'status', 'tagline',
       'title', 'vote_average', 'vote_count'],
      dtype='object')

In [202]:
movies[movies['vote_count']==0.0]['vote_average'].value_counts()

vote_average
0.0    2899
Name: count, dtype: int64

## Cleaning DateTime Columns

14. __Convert__ the datatype in the column __"release_date"__ __to datetime__. Set invalid values as NaN.

In [ ]:
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')


In [217]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 45466 entries, 0 to 45465
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   belongs_to_collection  45466 non-null  object        
 1   budget_musd            45466 non-null  float64       
 2   genres                 45466 non-null  object        
 3   id                     45463 non-null  float64       
 4   original_language      45455 non-null  object        
 5   overview               44512 non-null  object        
 6   popularity             45460 non-null  float64       
 7   poster_path            45080 non-null  object        
 8   production_companies   45466 non-null  object        
 9   production_countries   45466 non-null  object        
 10  release_date           45376 non-null  datetime64[ns]
 11  revenue_musd           45466 non-null  float64       
 12  runtime                45466 non-null  float64       
 13  spoken

## Cleaning Text / String Columns

15. __Analyze__ the text columns "overview" and "tagline". Try to identify __missing data that is not represented by NaN__ (e.g. "No Data"). __Replace as NaN__ (np.nan)!

In [218]:
movies['overview'].value_counts()

overview
No overview found.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       133
No Overview                                                                                                                                                                                                                       

In [219]:
movies['overview']= movies['overview'].replace(['No overview found.','No Overview','No movie overview available'],np.nan)

In [234]:
movies[movies['tagline'].str.contains('no ',case=False,na=False)]['tagline']

9                     No limits. No fears. No substitutes.
14       The Course Has Been Set. There Is No Turning B...
15                        No one stays at the top forever.
49            Five Criminals. One Line Up. No Coincidence.
50                                She's no angel of mercy.
                               ...                        
44569    Jungle holds no terrors for these bloodthirsty...
44901    Manga artist Yuichi Okano begins writing nurse...
44928        There is no TripAdvisor for this adventure...
45114                        Destiny... there is no escape
45364             She cried rape. But no one would listen.
Name: tagline, Length: 699, dtype: object

## Removing Duplicates

16. __Identify__ and __remove__ duplicates!

In [235]:
movies = movies.drop_duplicates()

## Handling Missing Values & Removing Observations

17. __Drop__ all rows/movies with unknown __id__ or __title__.

In [236]:
movies = movies.dropna(subset=['title','id'])

18. __Keep__ only those rows/movies in the df with __10 or more non-NaN__ values.

In [237]:
movies = movies.dropna(thresh=10)

## Final (Cleaning) Steps

19. __Keep__ only those rows/movies in the df with __status "Released"__. Then __drop__ the column "status".

In [238]:
movies= movies[movies['status']=='Released']
movies = movies.drop(columns=['status'])

20. The Order of the columns should be as follows: 

In [240]:
movies.columns

Index(['index', 'belongs_to_collection', 'budget_musd', 'genres', 'id',
       'original_language', 'overview', 'popularity', 'poster_path',
       'production_companies', 'production_countries', 'release_date',
       'revenue_musd', 'runtime', 'spoken_languages', 'tagline', 'title',
       'vote_average', 'vote_count'],
      dtype='object')

In [242]:
movies = movies[["id", "title", "tagline", "release_date", "genres", "belongs_to_collection", 
"original_language", "budget_musd", "revenue_musd", "production_companies",
"production_countries", "vote_count", "vote_average", "popularity", "runtime",
"overview", "spoken_languages", "poster_path"]]

21. __Reset__ the Index and create a __RangeIndex__.

In [239]:
movies = movies.reset_index()


22. __Save__ the cleaned dataset in a __csv-file__.

In [243]:
movies.to_csv('my_movies_clean.csv')

# +++++++++ See some Hints below +++++++++++++

# ++++++++++++++++ Hints++++++++++++++++++++

__Hints for 3.__ <br>
apply ast.literal_eval() on all stringified elements (you have to import ast):

In [ ]:
# example:
df.stringified_column = df.stringified_column.apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else np.nan)

__Hints for 4., 5., 6., 7., 8.__<br> 
apply an appropriate lambda function on all column elements

__Hints for 9.__<br>
Replace all __""__ (empty strings) in the above columns by NaN (__np.nan__)

__Hints for 10.__<br>
Use pd.to_numeric() and "coerce" errors

__Hints for 11.__<br>
Replace the value 0 by NaN (__np.nan__)

__Hints for 13.__<br>
Replace the value 0 by NaN (__np.nan__)

__Hints for 14.__<br>
Use pd.to_datetime() and "coerce" errors

__Hints for 16.__<br>
There cannot be two or more movies with the same movie id.